In [1]:
import pandas as pd

In [2]:
import rich

In [3]:
from transformers import AutoTokenizer

/dfs/data/uv-venv/modelscope/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [45]:
data_path="/dfs/data/work/hardtry/data/gem/gem_openai_messages_fc.json"
# data_path="/dfs/data/work/hardtry/data/hardgen/hardgen_openai_messages_fc.json"

In [46]:
df=pd.read_json(data_path,orient="records",lines=True)

In [6]:
tokenizer_path="/dfs/data/models/Qwen3-4B-Instruct-2507"

In [7]:
tokenizer=AutoTokenizer.from_pretrained(tokenizer_path)

In [14]:
def map_len(row):
    """
    适配pandas行级处理：row是df的一行（Series对象）
    """
    # 提取当前行的messages字段
    messages = row["messages"]
    # 用tokenizer处理messages，计算token长度
    prompt = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=False,
        tokenize=True
    )
    # 返回token长度（直接返回值，而非字典，更易整合到df）
    return len(prompt)

In [47]:
df["token_len"] = df.apply(map_len, axis=1)

In [48]:
rich.print(df["token_len"].describe())

count     5676.000000
mean      1978.018499
std        736.703902
min        631.000000
25%       1560.000000
50%       1888.000000
75%       2267.000000
max      19108.000000
Name: token_len, dtype: float64

In [59]:
df_filtered = df[df["token_len"] > 2048]
df_filtered = df_filtered[df_filtered["token_len"] < 10240]

In [60]:
rich.print(df_filtered["token_len"].describe())

count    2188.000000
mean     2541.526965
std       486.678387
min      2049.000000
25%      2203.000000
50%      2402.500000
75%      2715.000000
max      6101.000000
Name: token_len, dtype: float64

In [62]:
save_path = "/dfs/data/work/hardtry/data/gem/gem_openai_messages_fc_filtered.json"

df_filtered.to_json(
    save_path,
    orient="records",  # 按行生成JSON对象（每行对应df的一行）
    lines=True,        # 开启JSONL模式（每行一个JSON）
    force_ascii=False, # 保留中文（避免中文被转义为\uXXXX）
    index=False        # 不保存df的索引列（避免冗余）
)

In [63]:
df2=pd.read_json(save_path,orient="records",lines=True)

In [64]:
df2

,data_id,messages,token_len
0,49,"[{'role': 'system', 'content': 'You are a proj...",2228
1,330,"[{'role': 'system', 'content': 'You are a docu...",2051
2,199,"[{'role': 'system', 'content': 'You are a tabl...",2273
3,356,"[{'role': 'system', 'content': 'You are a rese...",2118
4,344,"[{'role': 'system', 'content': 'You are an AI ...",2612
...,...,...,...
2183,229712,"[{'role': 'system', 'content': 'You are a rese...",2601
2184,229951,"[{'role': 'system', 'content': 'You are a mark...",2356
2185,230080,"[{'role': 'system', 'content': 'You are a rese...",2423
2186,230534,"[{'role': 'system', 'content': 'You are an AI ...",2866
